# 04 — Run CalculiX Solver

This notebook runs `ccx` from Python, captures stdout/stderr, and lists generated solver files.
It fails loudly if CalculiX is not installed or not on `PATH`.

## What this notebook does

- Rebuilds the deterministic geometry and mesh.
- Checks whether `ccx` is available.
- Runs the solver from Python using `subprocess`.
- Captures stdout/stderr and lists `.dat`, `.frd`, `.sta`, and `.cvg` outputs.

In [4]:
import json
import logging
import sys
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

def find_module_root(start: Path | None = None) -> Path:
    current = start or Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate the fea_cad_one_sample module root.')

def json_safe(value: Any) -> Any:
    if is_dataclass(value):
        return json_safe(asdict(value))
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    return value

def write_json(path: Path, payload: Any) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return path

def load_json(path: Path) -> dict[str, Any]:
    return json.loads(Path(path).read_text(encoding='utf-8'))

MODULE_ROOT = find_module_root()
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import src.interfaces as iface

plt.style.use('seaborn-v0_8-whitegrid')
print(f'[SETUP] MODULE_ROOT={MODULE_ROOT}')

[SETUP] MODULE_ROOT=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample


In [5]:
import shutil

RUN_DIR = MODULE_ROOT / 'outputs' / 'sample_00689964' / '01_dataset_original'
SOURCE_STEP_PATH = RUN_DIR / "original.step" 
MESH_SIZE_MM = 8.0
LOAD_MAGNITUDE_N = 100.0

print('[STAGE] solver: preflight and run ccx')
ccx_path = shutil.which('ccx')
print('  ccx =', ccx_path)
if not ccx_path:
    message = "CalculiX executable 'ccx' was not found on PATH. Install ccx before running this notebook."
    print(f'ERROR: FileNotFoundError: {message}')
    raise FileNotFoundError(message)

config = iface.build_baseline_config(
    run_dir=RUN_DIR,
    source_step_path=SOURCE_STEP_PATH,
    mesh_size_mm=MESH_SIZE_MM,
    load_magnitude_n=LOAD_MAGNITUDE_N,
)
geometry = iface.prepare_geometry_artifacts(config, force=True)
mesh = iface.generate_calculix_mesh(config, geometry, force=True)
print('  input deck =', mesh.inp_path)
try:
    solver = iface.run_calculix_solver(config, mesh)
except Exception as exc:
    print(f'ERROR: {type(exc).__name__}: {exc}')
    raise

print('  stdout_path =', solver.stdout_path)
print('  stderr_path =', solver.stderr_path)
print('  dat_path =', solver.dat_path)
print('  frd_path =', solver.frd_path)
print('  sta_path =', solver.sta_path)
print('  cvg_path =', solver.cvg_path)
print('  generated files:')
for path in solver.output_files:
    print('   -', path)
print('  optional CGX viewer: cgx -v', solver.frd_path.name)
print('✓ solver finished')

INFO src.fea_replication.pipeline: build_baseline_config | start | run_dir=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original | source_step_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original/original.step | mesh_size_mm=8.0 | load_magnitude_n=100.0
INFO src.fea_replication.pipeline: build_baseline_config | done | run_dir=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original
INFO src.fea_replication.geometry: prepare_geometry_artifacts | start | run_dir=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original | source_step_path=/Users/vedaangchopr

[STAGE] solver: preflight and run ccx
  ccx = /opt/homebrew/Caskroom/miniconda/base/envs/cad_physics/bin/ccx


INFO src.fea_replication.geometry: load_geometry_summary | done | step_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original/cantilever_beam.step | major_axis=Y | spans_mm=(0.6185900000005435, 0.964536, 0.5357146545280003)
INFO src.fea_replication.geometry: prepare_geometry_artifacts | done | step_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original/cantilever_beam.step | preview_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/01_dataset_original/geometry_preview.png
INFO src.fea_replication.mesh: generate_calculix_mesh | start | step_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/ou

Info    :  - Label 'Shapes/Open CASCADE STEP translator 7.8 1' (3D)
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 10%] Meshing curve 3 (Line)
Info    : [ 20%] Meshing curve 4 (Line)
Info    : [ 20%] Meshing curve 5 (Line)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 30%] Meshing curve 7 (Line)
Info    : [ 40%] Meshing curve 8 (Line)
Info    : [ 40%] Meshing curve 9 (Line)
Info    : [ 50%] Meshing curve 10 (Line)
Info    : [ 50%] Meshing curve 11 (Line)
Info    : [ 60%] Meshing curve 12 (Line)
Info    : [ 60%] Meshing curve 13 (Line)
Info    : [ 70%] Meshing curve 14 (Circle)
Info    : [ 70%] Meshing curve 15 (Line)
Info    : [ 80%] Meshing curve 16 (Line)
Info    : [ 80%] Meshing curve 17 (Line)
Info    : [ 90%] Meshing curve 18 (Line)
Info    : [ 90%] Meshing curve 19 (Line)
Info    : [100%] Meshing curve 20 (Line)
Info    : [100%] Meshing curve 21 (Circle)
Info    : Done meshing 1D (Wall 0.000474084s, CPU 0.